In [3]:
import os
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from sklearn.svm import SVC

for i in range(1, 9):
    folder_path = os.path.join('initial_data', f'function_{i}')
    inputs_path = os.path.join(folder_path, 'initial_inputs.npy')
    outputs_path = os.path.join(folder_path, 'initial_outputs.npy')
    
    if not os.path.exists(inputs_path) or not os.path.exists(outputs_path):
        continue
        
    X_f = np.load(inputs_path)
    y_f = np.load(outputs_path)
    
    dim = X_f.shape[1] if len(X_f.shape) > 1 else 1
    
    if i == 5:
        beta = 0.05
    elif i in [1, 2]:
        beta = 3.2
    else:
        beta = 1.0
        
    np.random.seed(500 + i)
    X_grid = np.random.uniform(0.0, 1.0, size=(50000, dim))
    
    try:
        kernel = Matern(length_scale=[0.2] * dim, nu=2.5)
        gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-4, normalize_y=True, optimizer=None)
        gp.fit(X_f, y_f)
        
        if i not in [1, 2] and len(y_f) >= 15:
            threshold = np.percentile(y_f, 75)
            y_labels = np.where(y_f >= threshold, 1, 0)
            
            if len(np.unique(y_labels)) > 1:
                svm = SVC(kernel='rbf', C=1.0, gamma='scale')
                svm.fit(X_f, y_labels)
                preds = svm.predict(X_grid)
                X_filtered = X_grid[preds == 1]
                
                if len(X_filtered) > 100:
                    X_grid = X_filtered
        
        mean, sigma = gp.predict(X_grid, return_std=True)
        ucb_values = mean + beta * sigma
        best_idx = np.argmax(ucb_values)
        next_point = X_grid[best_idx]
        
    except Exception:
        from scipy.spatial.distance import cdist
        distances = cdist(X_grid, X_f)
        min_distances = np.min(distances, axis=1)
        best_idx = np.argmax(min_distances)
        next_point = X_grid[best_idx]
    
    portal_string = "-".join([f"{val:.6f}" for val in next_point])
    print(portal_string)

0.675477-0.000160
0.000030-0.854635
0.355006-0.487900-0.410307
0.454234-0.459940-0.398980-0.282405
0.339088-0.831659-0.964642-0.972950
0.597034-0.264056-0.622372-0.816516-0.068382
0.114639-0.425485-0.190218-0.121015-0.288420-0.784497
0.190188-0.181424-0.103023-0.288040-0.295864-0.802807-0.427642-0.776255
